In [ ]:
import torch
import json
import gc
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_LIST = [
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "Qwen/Qwen1.5-1.8B",
    "microsoft/phi-2"
]

MAX_LENGTH = 512
SAVE_EVERY = 500
DATA_PATH = "outputs_20_per_prompt.jsonl"

In [ ]:
def load_jsonl_flat(path):
    pairs = []
    with open(path, "r") as f:
        for line in f:
            item = json.loads(line)
            pairs.append((item["question"], item["output"]))
    return pairs

pairs = load_jsonl_flat(DATA_PATH)

print(f"Total pairs: {len(pairs)}")

In [ ]:
def compute_frost_score(model, tokenizer, fisher, prompt, response):

    text = prompt + "\n" + response

    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    )

    input_ids = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)

    labels = input_ids.clone()
    labels[attention_mask == 0] = -100

    num_tokens = (labels != -100).sum().item()

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

    loss = outputs.loss * num_tokens

    model.zero_grad(set_to_none=True)
    loss.backward()

    score = 0.0
    eps = 1e-8

    for name, param in model.named_parameters():
        if param.grad is not None:
            grad_sq = param.grad.detach().float() ** 2
            score += (grad_sq / (fisher[name] + eps)).sum()

    return score.item()

In [ ]:
all_results = {}

for model_name in MODEL_LIST:

    print(f"\nProcessing model: {model_name}")

    # Load tokenizer + model
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
    ).to(DEVICE)

    model.eval()

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Load Fisher (move to GPU ONCE)
    fisher_path = f"fisher_diag_{model_name.replace('/', '_')}.pt"
    fisher = torch.load(fisher_path)
    fisher = {k: v.to(DEVICE) for k, v in fisher.items()}

    results = []

    save_path = f"frost_scores_{model_name.replace('/', '_')}.json"

    for i, (prompt, response) in enumerate(tqdm(pairs)):

        score = compute_frost_score(
            model, tokenizer, fisher, prompt, response
        )

        results.append({
            "prompt": prompt,
            "response": response,
            "score": score
        })

        # checkpoint saving
        if (i + 1) % SAVE_EVERY == 0:
            with open(save_path, "w") as f:
                json.dump(results, f)

    # Final save
    with open(save_path, "w") as f:
        json.dump(results, f)

    print(f"Saved: {save_path}")

    all_results[model_name] = results

    # Cleanup GPU memory
    del model
    del tokenizer
    torch.cuda.empty_cache()
    gc.collect()